# Tahoe DEM Mosaic

Build one bare-earth DEM (land surface plus lake bottom) from six inputs of different vintage,
sensor, and resolution. At every cell the best available source wins, chosen by a per-zone
priority list with NoData fall-through, and a source-ID raster records which one.

| Source | Type | Native cell | CRS / units | Role |
|---|---|---|---|---|
| 2022 USGS OPR tiles, zone 10 (mosaic dataset) | topo | 0.5 m | NAD83(2011) UTM 10N, m | Primary land surface, vertical reference |
| 2022 USGS OPR tiles, zone 11 (mosaic dataset) | topo | 0.5 m | NAD83(2011) UTM 11N, m | Primary land surface east of the 120th meridian |
| 2022 USGS seamless 1 m DEM, ten zone-10 tiles (mosaic dataset) | topo | 1 m | NAD83 UTM 10N, m | Patches the two wedges along the meridian that neither OPR work unit tiles |
| DEM_BareEarth_LiDAR_2010 (SDE) | topo | 2 m | UTM 10N, m | Fills remaining 2022 voids and coverage gaps |
| Nearshore_BareEarth_DEM, 2018 green lidar (SDE; config key `green_2020`) | topo + shallow bathy | 0.5 m | UTM 10N, m | Nearshore bottom to ~13 m depth, shoreline transition |
| DEM_USGS_DeepWaterBathyTopo, 1998 sonar, USGS DDS-55 (SDE) | deep bathy + topo skin | 10 m | UTM 11N, ft | Deep lake bottom |

The 2022 lidar enters as two OPR sources, one per UTM zone, so each is resampled exactly once
by this pipeline. The earlier single SDE raster was a cross-zone resample at 0.728 x 0.676 m
whose row and column moire hillshaded as terracing. The two OPR work units are tiled so that
two wedges along 120 W fall in neither; USGS's seamless 1 m DEM for the same project covers
them and is ranked just below the OPR halves. Build the three mosaic datasets first with
`scripts/build_2022_source.py` (`--tiles` for the OPR folder, `--onem-tiles` for the ten 1 m
tiles listed in `data/usgs_2022_1m_wedge_patch_urls.txt`). Full source citations are in
`config.yaml` under `metadata.source_citations`.

SDE is read only: nothing here writes to it.

**This notebook is a front end for `../scripts/build_dem_mosaic.py`.** All logic lives in the
script so the two cannot drift; the notebook exists to run steps one at a time and look at the
tables. For the full basin, run the script from a terminal on the server instead of here, since
it takes hours and a dropped kernel loses the run:

```
python scripts/build_dem_mosaic.py --steps 1-3 --test    # minutes: check the datum solution first
python scripts/build_dem_mosaic.py --test                 # whole pipeline on the test box
python scripts/build_dem_mosaic.py                        # full basin
```

Every step reuses outputs already in the scratch gdb, so a crashed run resumes where it stopped.
`--force` rebuilds the selected steps. Intermediates are dropped as each step consumes them;
`--keep` retains them.

## Things that will bite you

**Vertical datum.** A 0.5 m datum mismatch is a cliff along every seam. Step 3 solves it: the
zone-10 2022 lidar is the reference, and each other source is shifted by the negated median of
its difference against an already-aligned source over stable ground. Read `overlap_qa.csv`
before trusting the result: `LARGE_SHIFT` is expected for a datum change and fine; `TILT` means
the difference varies across the overlap and a constant shift will not fix it; `UNSOLVED` means
too few usable samples. The east-versus-west pair is the same product in two zones with a
sliver of overlap, so it is diagnostic only and its offset is fixed at zero in the config. The
1 m patch overlaps the zone-10 half broadly, so its pair solves properly and should come out near zero.

**Terrestrial lidar over water is not ground.** The USGS OPR tiles, the 1 m patch, and the 2010
lidar are all hydro-flattened, holding one constant value across the lake. Step 5 detects each
product's water-surface elevation from its own values over the open lake and strips everything
inside the high-water polygon at or below it, keeping exposed beach. No gauge data needed.

**The high-water polygon is a zone boundary and extent, not a mask** for the terrestrial lidar.
The lake was below the legal maximum during every acquisition, so real beach sits inside it.

**Do not call the deep lake 1 m data.** The output grid is 1 m because the lidar earns it; the
deep lake is resampled 10 m sonar, and shallower than about 13 m the USGS grid diverges from
the 2018 lidar with depth for reasons the product does not document. The source-ID raster is
how downstream users know which is which. Publish it with the DEM.

**The inputs are large.** The script sets the output extent before projecting so only the basin
is resampled. `test_extent` in the config runs everything on a small box in minutes.

**2010 vs 2022 differences are partly real.** Fire, construction, and beach migration show up.
The offset solver uses the median on low-slope land so that real change is outvoted.

## 0. Setup

`test=True` uses `target.test_extent` from the config and keeps all intermediates under a
`dm_test_` prefix so nothing collides with a full-basin run in the same scratch gdb.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
sys.path.insert(0, str(ROOT / "scripts"))
from build_dem_mosaic import Pipeline

p = Pipeline(ROOT / "config.yaml", test=True, force=False)

## 1. Inventory

What each file says about itself. `v_crs` is empty for all four, which is why `v_datum` and
`z_units` are required in the config. `z_units_guess` checks the config against the value range:
Tahoe elevations are roughly 1,400 to 3,300 m or 4,500 to 11,000 ft, so the two are unambiguous.

In [ ]:
inv = p.step1_inventory()
inv.T

## 2. Standardize onto the target grid

Per source: bathy is focal-mean smoothed at native resolution first, then `ProjectRaster` with
bilinear resampling at the target cell. The 2022 lidar defines the grid origin (registration
point 0,0); every other source snaps to it. Feet become meters. Values outside the plausible
range are nulled. Vertical offsets are **not** applied here so a changed offset never forces a
re-projection. The log shows which geographic transformation was used for the UTM 11N bathy.

In [ ]:
p.step2_standardize()

## 3. Solve vertical offsets

For each `[source, against]` pair in `qa.offset_chain`, sample the difference raster over
stable ground and take the median. Land pairs are restricted to outside the lake polygon and
slopes below `max_slope_deg`; the underwater pair (sonar vs green lidar) is restricted to the
lake. A plane is fitted to the differences to detect tilt.

What to look for:

- `median` near 0, tight `iqr`: datums already agree.
- `median` clearly nonzero, tight `iqr`: a systematic datum or geoid offset. This is what the
  solver fixes. `LARGE_SHIFT` just flags it for your attention.
- wide `iqr`: noise or real change. The median still holds if most of the overlap is unchanged.
- `TILT`: the offset varies spatially (different geoid model, or a tilted product). A constant
  shift will not fix it. Open the `dm_*diff_*` raster in Pro to see the pattern.
- `UNSOLVED`: too few usable samples. Widen the extent, relax `max_slope_deg`, or set the
  offset manually in the config.

Solved offsets go to `resolved_offsets.yaml` in the outputs folder. Sources with
`vertical_offset_m: auto` in the config pick them up in step 5.

In [ ]:
qa = p.step3_overlap_qa()
qa

In [ ]:
# Sanity check the chain: sonar was aligned to green, green to 2022. Offsets are relative to 2022.
import yaml
yaml.safe_load(p.resolved_offsets_file().read_text())

## 4. Zone raster

1 = land (inside AOI, outside lake polygon), 2 = water (inside lake polygon). Every raster from
here on is full-extent, and a source that does not cover a cell is NoData there. That is what
makes the priority fall-through in step 6 work.

In [ ]:
p.step4_zones()

## 5. Apply offsets and clean

- **Terrestrial lidar**: the water surface is the median elevation inside the lake polygon eroded
  inward by `offshore_erode_m`. MAD near zero confirms a flattened product (2010); a few
  decimeters is the expected 2022 noise; larger means the sample is contaminated and the erosion
  distance should grow. Cells inside the lake polygon at or below `surface + water_strip_tol_m`
  are nulled. The detected 2010 and 2022 surfaces should differ by the stage change between
  flights plus the datum offset step 3 found; `water_surface_aligned_m` shows them after
  alignment, so the remaining difference is stage alone.
- **Green lidar**: optional absolute elevation floor.
- **Sonar**: restricted to the lake polygon, which discards its topo skin.

In [ ]:
ws = p.step5_clean()
ws

## 6. Priority mosaic

First valid source wins, per zone, plus a source-ID raster. Feathering (`blend.feather_m`)
ramps each winner into its fallback over that distance from the winner's data edge; it hides
steps but also smears real edges and costs a Euclidean distance pass per source. Leave it at 0
until the datum work is done, because feathering a datum error only hides evidence.

In [ ]:
p.step6_mosaic()

## 7. Export COGs, hillshade, provenance

In [ ]:
prov = p.step7_export()
prov

## 8. Result QA

Area by source (the sonar should never win on land), NoData holes inside the AOI, and the 3x3
elevation range at cells where the source ID changes versus everywhere else. A seam median much
larger than the background median means a residual offset. For visual checks open the hillshade
in Pro along the shoreline and along the 2022/2010 coverage edge, and run Stack Profile across
a few beaches.

In [ ]:
area, n_holes, seams = p.step8_result_qa()
print(f"holes inside AOI: {n_holes} cells")
display(area)
seams